In [0]:
spark

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
conn = w.connections.get("earthquake_connection")
base_url = f"{conn.options['host']}{conn.options['base_path']}"

In [0]:
dbutils.widgets.text("catalog_name", "earthquake_catalog_dev", "earthquake_catalog_dev")
catalog_name = dbutils.widgets.get("catalog_name")
print(catalog_name)

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql("USE SCHEMA bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS earthquake_data")

In [0]:
import requests
import json
import datetime

url = f"{base_url}summary/all_day.geojson"
response = requests.get(url)
data = response.json()
if response.status_code != 200:
    raise Exception(f"Error in getting the data from {url}")
dt = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{dt}.json",
    json.dumps(data),
    overwrite=True,
)